# 02 Plate Solving

**Eingabe**: kalibriertes Referenzbild (erster Frame der Nacht, bestes Seeing)  
**Ausgabe**: FITS mit WCS-Header (RA/Dec ↔ Pixel)

Werkzeug: **ASTAP-CLI** (`astap -f <datei> -r 10 -wcs`)  
Fallback: `astrometry.net` (online oder lokal)

ASTAP muss separat installiert sein (hnsky.org, Apple Silicon Signing beachten).
Datenbank D05 oder D80 herunterladen.

Flags aus `astap -h` prüfen — KONZEPT.md ADR-01 und §8 (Test-Status: ungetestet).

In [ ]:
import subprocess, json
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS

ASTAP = "/Applications/astap"  # Pfad anpassen (Apple Silicon: Signing beachten)

# session_config.json aus Notebook 01 laden
_cfgs = list(Path.home().rglob("session_config.json"))
if not _cfgs:
    raise FileNotFoundError("session_config.json nicht gefunden — bitte Notebook 01 ausführen.")
session_cfg = json.loads(_cfgs[0].read_text())
WORKSPACE = Path(session_cfg["workspace"])
DIR_CAL   = Path(session_cfg["dir_cal"])

_FITS_GLOB = ["*.fits", "*.fit", "*.fts", "*.FITS", "*.FIT", "*.FTS"]
cal_frames = sorted(f for g in _FITS_GLOB for f in DIR_CAL.glob(g))
if not cal_frames:
    raise FileNotFoundError(f"Keine FITS in {DIR_CAL} — Siril-Kalibrierung abgeschlossen?")
REF = cal_frames[0]

print(f"Workspace:          {WORKSPACE}")
print(f"Kalibrierte Bilder: {len(cal_frames)} in {DIR_CAL}")
print(f"Referenzbild:       {REF}")

In [ ]:
# ASTAP aufrufen (Flags mit `astap -h` verifizieren)
result = subprocess.run(
    [ASTAP, "-f", str(REF), "-r", "10", "-wcs"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "(keine Ausgabe)")
if result.returncode != 0:
    print("FEHLER:", result.stderr[-300:])

In [ ]:
# WCS prüfen
with fits.open(REF) as hdul:
    w = WCS(hdul[0].header)
print("WCS:", w)